# The five Anthropic workflow patterns

Adapted from ["Building effective agents"](https://www.anthropic.com/research/building-effective-agents). Each is a *composition pattern* over LLM calls, not a framework feature.

1. **Prompt chain** — step A's output feeds step B.
2. **Routing** — classify the input, dispatch to a specialized prompt.
3. **Parallelization** — run N attempts concurrently, then pick / aggregate.
4. **Orchestrator-workers** — one call decomposes, fan-out workers handle each piece.
5. **Evaluator-optimizer** — draft, evaluate, revise, repeat until good enough.

In [1]:
# %% Notebook bootstrap — never touches API keys directly.
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# CI / offline mode: replay cached responses instead of calling out.
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
print('LLM_CACHE_ONLY =', os.environ.get('LLM_CACHE_ONLY', '0'))


LLM_CACHE_ONLY = 1


In [2]:
import asyncio, json
from shared.llm import Message, complete

MODEL = 'openai/gpt-4o-mini'
NS = '00-foundations/04-prompt-patterns'

ABSTRACT = (
    'We present RA-MoE, a sparse mixture-of-experts architecture in which router '
    'logits are biased toward experts whose recent activation history minimizes '
    'inter-device communication. On a 47B-parameter language model with 32 experts '
    'and top-2 routing, RA-MoE reduces p99 decode latency by 38% relative to '
    'standard learned routing while keeping perplexity within 0.4% of the baseline.'
)

## 1) Prompt chain — summarize, then critique the summary

In [3]:
summary = complete(
    model=MODEL, namespace=NS,
    messages=[
        Message(role='system', content='Summarize the abstract in one tight sentence.'),
        Message(role='user', content=ABSTRACT),
    ],
).content
print('SUMMARY:', summary)

critique = complete(
    model=MODEL, namespace=NS,
    messages=[
        Message(role='system', content='Critique the following summary in one sentence: does it preserve the key numeric claim?'),
        Message(role='user', content=summary),
    ],
).content
print('CRITIQUE:', critique)

SUMMARY: RA-MoE biases MoE router logits by recent expert activation history to cut p99 decode latency 38% with minimal perplexity impact.
CRITIQUE: Yes — the 38% p99 latency reduction is preserved and attribution to the routing-history bias is intact.


## 2) Routing — classify the request, dispatch to a specialized prompt

In [4]:
def route(query: str) -> str:
    return complete(
        model=MODEL, namespace=NS,
        messages=[
            Message(role='system', content='Classify the request as exactly one of: SUMMARIZE, TRANSLATE, EXPLAIN. Reply with only the label.'),
            Message(role='user', content=query),
        ],
    ).content.strip()

HANDLERS = {
    'SUMMARIZE': 'Summarize the user\'s text in one short sentence.',
    'TRANSLATE': 'Translate the user\'s text into French.',
    'EXPLAIN': 'Explain the user\'s text as if to a smart 12-year-old.',
}

intent = route('Give me a one-sentence summary of this paper.')
print('intent:', intent)
handler_system = HANDLERS[intent]
out = complete(
    model=MODEL, namespace=NS,
    messages=[
        Message(role='system', content=handler_system),
        Message(role='user', content=ABSTRACT),
    ],
).content
print('OUT:', out)

intent: SUMMARIZE
OUT: RA-MoE cuts MoE decode latency by 38% via activation-history-biased routing.


## 3) Parallelization — N attempts, then judge
Drafting N candidates in parallel costs `max(latency_i)` instead of `sum(latency_i)`. A small judge call picks the best — cheaper and more reliable than top-p alone.

In [5]:
async def draft(i: int) -> str:
    # In offline cache-only mode this still runs synchronously per call; the async
    # shape shows the pattern.
    return complete(
        model=MODEL, namespace=NS, temperature=0.7,
        messages=[
            Message(role='system', content=f'Write a one-sentence summary, attempt #{i+1}.'),
            Message(role='user', content=ABSTRACT),
        ],
    ).content

async def candidates() -> list[str]:
    return await asyncio.gather(*(draft(i) for i in range(3)))

cands = await candidates()
for i, c in enumerate(cands):
    print(i, c)

0 RA-MoE routes experts using activation history, cutting p99 latency 38% at near-identical perplexity.
1 By biasing the router with recent expert utilization, RA-MoE reduces p99 decode latency by 38% on a 47B MoE.
2 RA-MoE adds no parameters yet drops p99 latency 38% by steering routing toward recently used experts.


In [6]:
judge_prompt = (
    'Pick the best one-sentence summary of the abstract. Reply with ONLY the integer index (0, 1, or 2).\n'
    'Abstract:\n' + ABSTRACT + '\n\nCandidates:\n'
    + '\n'.join(f'{i}) {c}' for i, c in enumerate(cands)) + '\n'
)
pick = complete(
    model=MODEL, namespace=NS,
    messages=[
        Message(role='system', content='You are an impartial judge.'),
        Message(role='user', content=judge_prompt),
    ],
).content.strip()
print('judge picked:', pick)
print('winner:', cands[int(pick)])

judge picked: 1
winner: By biasing the router with recent expert utilization, RA-MoE reduces p99 decode latency by 38% on a 47B MoE.


## 4) Orchestrator-workers — decompose, fan-out, gather
An orchestrator call decides *which* subtasks to do; workers execute them in parallel; the orchestrator (or a final assembler) stitches the result.

In [7]:
topics_raw = complete(
    model=MODEL, namespace=NS,
    messages=[
        Message(role='system', content='List 2 short bullet topics worth expanding from this abstract. One per line, no numbering.'),
        Message(role='user', content=ABSTRACT),
    ],
).content
topics = [t.strip() for t in topics_raw.splitlines() if t.strip()]
print('topics:', topics)

async def worker(topic: str) -> str:
    return complete(
        model=MODEL, namespace=NS,
        messages=[
            Message(role='system', content=f'Expand the topic \'{topic}\' into one tight paragraph grounded in the abstract.'),
            Message(role='user', content=ABSTRACT),
        ],
    ).content

sections = await asyncio.gather(*(worker(t) for t in topics))
for t, s in zip(topics, sections):
    print(f'### {t}\n{s}\n')

topics: ['The routing-history bias mechanism', 'The latency vs perplexity tradeoff']
### The routing-history bias mechanism
RA-MoE maintains an exponential moving average of per-expert activations and subtracts it from the router logits, steering top-2 routing toward experts whose recent use minimizes inter-device communication.

### The latency vs perplexity tradeoff
On a 47B MoE with 32 experts and top-2 routing, RA-MoE cuts p99 decode latency by 38% while perplexity stays within 0.4% of the standard learned router — essentially free throughput.



## 5) Evaluator-optimizer — draft, evaluate, revise
The evaluator returns a structured score so we know whether to loop again. In production, cap the number of iterations so a stubborn task can't run forever.

In [8]:
first_draft = complete(
    model=MODEL, namespace=NS,
    messages=[
        Message(role='system', content='Draft a one-sentence summary of the abstract.'),
        Message(role='user', content=ABSTRACT),
    ],
).content
print('DRAFT:', first_draft)

score_raw = complete(
    model=MODEL, namespace=NS,
    messages=[
        Message(role='system', content='Score the following one-sentence summary from 1-5 on faithfulness AND specificity. Reply with a JSON object {"faithfulness": int, "specificity": int, "feedback": str}.'),
        Message(role='user', content=f'Abstract:\n{ABSTRACT}\n\nSummary:\n{first_draft}'),
    ],
    response_format={'type': 'json_object'},
).content
score = json.loads(score_raw)
print('SCORE:', score)

revised = complete(
    model=MODEL, namespace=NS,
    messages=[
        Message(role='system', content='Revise the summary to address the feedback. Return only the revised one-sentence summary.'),
        Message(role='user', content=f'Abstract:\n{ABSTRACT}\n\nDraft:\n{first_draft}\n\nFeedback: ' + score['feedback']),
    ],
).content
print('REVISED:', revised)

DRAFT: RA-MoE reduces latency for sparse MoE inference.
SCORE: {'faithfulness': 4, 'specificity': 2, 'feedback': 'Misses the 38% p99 number and the routing-history mechanism.'}
REVISED: RA-MoE cuts p99 decode latency by 38% on a 47B MoE by biasing the router toward recently activated experts, with under 0.4% perplexity impact.


## Recap

| Pattern | Calls (typical) | When to reach for it |
|---|---|---|
| Prompt chain | 2-3 | One step's output is the next step's input |
| Routing | 2 | Many handlers, only one fires per request |
| Parallelization | N + 1 | Quality benefits from candidate diversity |
| Orchestrator-workers | 1 + N | Subtasks are independent and known up front |
| Evaluator-optimizer | 2-K | Quality must clear a measurable bar |

These compose. Real agents stack several of them; the agentic-framework leaves in `03-agentic-frameworks/` show the same patterns implemented with first-class library support.